In [26]:
#ЯЧЕЙКА 1 — Установка библиотек
!pip install -q transformers datasets evaluate seqeval accelerate sentencepiece

In [27]:
#ЯЧЕЙКА 2 — Импорты
import numpy as np
import pandas as pd
from datasets import Dataset, load_dataset, DatasetDict


from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForMaskedLM,
    DataCollatorForTokenClassification,
    DataCollatorForWholeWordMask,
    TrainingArguments,
    Trainer,
    pipeline
)

import evaluate
import torch

In [28]:
# Загружаем датасет factRuEval-2016
dataset = load_dataset("gusevski/factrueval2016")

train_data = dataset["train"][0]["data"]
valid_data = dataset["validation"][0]["data"]
test_data = dataset["test"][0]["data"]

dataset = DatasetDict({
    "train": Dataset.from_list(train_data),
    "validation": Dataset.from_list(valid_data),
    "test": Dataset.from_list(test_data),
})

dataset

Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 7746
    })
    validation: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 2582
    })
    test: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 2582
    })
})

In [29]:
# Посмотрим пример
dataset["train"][0]

{'id': 0,
 'tokens': ['"',
  'Если',
  'Миронов',
  'занял',
  'столь',
  'оппозиционную',
  'позицию',
  ',',
  'то',
  'мне',
  'представляется',
  ',',
  'что',
  'для',
  'него',
  'было',
  'бы',
  'порядочным',
  'и',
  'правильным',
  'уйти',
  'в',
  'отставку',
  'с',
  'занимаемого',
  'им',
  'поста',
  ',',
  'поста',
  ',',
  'который',
  'предоставлен',
  'ему',
  'сегодня',
  '"',
  'Единой',
  'Россией',
  "''",
  'и',
  'никем',
  'больше',
  "''",
  ',',
  '-',
  'заключает',
  'Исаев',
  '.'],
 'length': 47,
 'ner_tags_str': ['O',
  'O',
  'B-PER',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-ORG',
  'I-ORG',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-PER',
  'O'],
 'ner_tags': [0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0

In [30]:
all_labels = set()

for sample in dataset["train"]:
    all_labels.update(sample["ner_tags_str"])

label_names = sorted(list(all_labels))

print(label_names)


num_labels = len(label_names)

print("Количество labels:", num_labels)

['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']
Количество labels: 7


In [31]:
#ЯЧЕЙКА 6 — Загружаем токенизатор
MODEL_NAME = "DeepPavlov/rubert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [32]:
#ЯЧЕЙКА 7 — Подготовка данных и выравнивание меток
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}

print(label2id)

{'B-LOC': 0, 'B-ORG': 1, 'B-PER': 2, 'I-LOC': 3, 'I-ORG': 4, 'I-PER': 5, 'O': 6}


In [33]:
sample = dataset["train"][0]

for token, tag_id, tag_str in zip(
    sample["tokens"],
    sample["ner_tags"],
    sample["ner_tags_str"]
):
    print(f"{token:15} {tag_id} {tag_str}")

"               0 O
Если            0 O
Миронов         1 B-PER
занял           0 O
столь           0 O
оппозиционную   0 O
позицию         0 O
,               0 O
то              0 O
мне             0 O
представляется  0 O
,               0 O
что             0 O
для             0 O
него            0 O
было            0 O
бы              0 O
порядочным      0 O
и               0 O
правильным      0 O
уйти            0 O
в               0 O
отставку        0 O
с               0 O
занимаемого     0 O
им              0 O
поста           0 O
,               0 O
поста           0 O
,               0 O
который         0 O
предоставлен    0 O
ему             0 O
сегодня         0 O
"               0 O
Единой          3 B-ORG
Россией         4 I-ORG
''              0 O
и               0 O
никем           0 O
больше          0 O
''              0 O
,               0 O
-               0 O
заключает       0 O
Исаев           1 B-PER
.               0 O


In [34]:
#ЯЧЕЙКА 8 — Функция токенизации и alignment
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding=False
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:

            # Специальные токены
            if word_idx is None:
                label_ids.append(-100)

            # Первый токен слова
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            # Остальные subword токены
            else:
                # Можно:
                # либо повторять label
                # либо ставить -100
                label_ids.append(label[word_idx])

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

In [35]:
#ЯЧЕЙКА 9 — Применяем preprocessing
tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

tokenized_datasets

Map:   0%|          | 0/7746 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 7746
    })
    validation: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2582
    })
    test: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2582
    })
})

In [36]:
#ЯЧЕЙКА 10 — Data Collator
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [37]:
#ЯЧЕЙКА 11 — Метрики NER
metric = evaluate.load("seqeval")

In [38]:
#ЯЧЕЙКА 12 — Функция вычисления метрик
def compute_metrics(p):

    predictions, labels = p

    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):

        current_predictions = []
        current_labels = []

        for p_, l_ in zip(prediction, label):

            if l_ != -100:
                current_predictions.append(label_names[p_])
                current_labels.append(label_names[l_])

        true_predictions.append(current_predictions)
        true_labels.append(current_labels)

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [39]:
#ЯЧЕЙКА 13 — Baseline без fine-tuning
baseline_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                        

In [40]:
#ЯЧЕЙКА 14 — Baseline evaluation
baseline_args = TrainingArguments(
    output_dir="./baseline_eval",
    per_device_eval_batch_size=8,
    do_eval=True
)

baseline_trainer = Trainer(
    model=baseline_model,
    args=baseline_args,
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

baseline_metrics = baseline_trainer.evaluate()

print(baseline_metrics)

{'eval_loss': 1.967424750328064, 'eval_model_preparation_time': 0.0026, 'eval_precision': 0.23135285549685863, 'eval_recall': 0.18031786912467382, 'eval_f1': 0.20267194835121935, 'eval_accuracy': 0.14618759406923496, 'eval_runtime': 8.8992, 'eval_samples_per_second': 290.137, 'eval_steps_per_second': 36.295}


In [41]:
#ЯЧЕЙКА 15 — Fine-tuning NER модели
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                        

In [42]:
#ЯЧЕЙКА 16 — Параметры обучения
training_args = TrainingArguments(
    output_dir="./rubert_ner",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=100,

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    do_train=True,
    do_eval=True
)

In [43]:
#ЯЧЕЙКА 17 — Trainer
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [44]:
#ЯЧЕЙКА 18 — Обучение
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.027712,0.027670,0.992937,0.993703,0.993320,0.992898
2,0.022460,0.021318,0.995511,0.995895,0.995703,0.995420
3,0.005960,0.022878,0.994798,0.996264,0.995530,0.995238


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2907, training_loss=0.03293052494351627, metrics={'train_runtime': 464.5181, 'train_samples_per_second': 50.026, 'train_steps_per_second': 6.258, 'total_flos': 565674868981068.0, 'train_loss': 0.03293052494351627, 'epoch': 3.0})

In [45]:
#ЯЧЕЙКА 19 — Финальная оценка
finetuned_metrics = trainer.evaluate(
    tokenized_datasets["test"]
)

print(finetuned_metrics)


{'eval_loss': 0.020568475127220154, 'eval_precision': 0.9955084745762712, 'eval_recall': 0.995204852756786, 'eval_f1': 0.9953566405124642, 'eval_accuracy': 0.9954686649373939, 'eval_runtime': 9.1199, 'eval_samples_per_second': 283.117, 'eval_steps_per_second': 35.417, 'epoch': 3.0}


In [46]:
#ЯЧЕЙКА 20 — Сравнение baseline vs fine-tuned
comparison = pd.DataFrame({
    "Baseline": baseline_metrics,
    "Fine-tuned": finetuned_metrics
})

comparison

,Baseline,Fine-tuned
eval_loss,1.967425,0.020568
eval_model_preparation_time,0.002600,NaN
eval_precision,0.231353,0.995508
eval_recall,0.180318,0.995205
eval_f1,0.202672,0.995357
eval_accuracy,0.146188,0.995469
eval_runtime,8.899200,9.119900
eval_samples_per_second,290.137000,283.117000
eval_steps_per_second,36.295000,35.417000
epoch,NaN,3.000000


ЧАСТЬ 2 — MLM PRETRAINING

Теперь улучшим модель дополнительным MLM обучением.

In [69]:
#ЯЧЕЙКА 21 — Подготовка текстов
def join_tokens(example):
    return {
        "text": " ".join(example["tokens"])
    }

mlm_dataset = dataset.map(join_tokens)


#ЯЧЕЙКА 22 — Удаляем лишние колонки
columns_to_remove = mlm_dataset["train"].column_names

print(columns_to_remove)

Map:   0%|          | 0/7746 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags', 'text']


In [70]:
#ЯЧЕЙКА 22 — Токенизация для MLM
def tokenize_mlm(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        return_special_tokens_mask=True
    )

tokenized_mlm = mlm_dataset.map(
    tokenize_mlm,
    batched=True,
    remove_columns=columns_to_remove
)



Map:   0%|          | 0/7746 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

In [68]:
#tokenized_mlm["train"][0]

In [71]:
#ЯЧЕЙКА 23 — Whole Word Masking (не получилось)
from transformers import DataCollatorForLanguageModeling

wwm_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

In [72]:
#ЯЧЕЙКА 24 — Загружаем MLM модель
mlm_model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME
)

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                          | Status     |  | 
-----------------------------+------------+--+-
cls.seq_relationship.bias    | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight  | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be igno

In [73]:
#ЯЧЕЙКА 25 — Аргументы MLM обучения
mlm_args = TrainingArguments(
    output_dir="./rubert_mlm",

    learning_rate=5e-5,

    per_device_train_batch_size=8,

    num_train_epochs=2,

    save_strategy="epoch",

    logging_steps=100,

    do_train=True,

    report_to="none"
)

In [74]:
tokenized_mlm.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "token_type_ids",
        "special_tokens_mask"
    ]
)

In [75]:
#ЯЧЕЙКА 26 — MLM Trainer
mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_args,

    train_dataset=tokenized_mlm["train"],

    processing_class=tokenizer,

    data_collator=wwm_collator
)

In [76]:
#ЯЧЕЙКА 27 — MLM обучение
mlm_trainer.train()

Step,Training Loss
100,1.624197
200,1.844884
300,1.886481
400,1.946940
500,1.915513
600,1.844180
700,1.888955
800,1.854707
900,1.866174
1000,1.883605


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1938, training_loss=1.7548588220303027, metrics={'train_runtime': 551.7039, 'train_samples_per_second': 28.08, 'train_steps_per_second': 3.513, 'total_flos': 787552004619648.0, 'train_loss': 1.7548588220303027, 'epoch': 2.0})

In [77]:
#ЯЧЕЙКА 28 — Сохраняем MLM checkpoint
mlm_trainer.save_model("./rubert_mlm_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [78]:
#ЯЧЕЙКА 29 — Загружаем MLM checkpoint для NER
mlm_ner_model = AutoModelForTokenClassification.from_pretrained(
    "./rubert_mlm_final",

    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: ./rubert_mlm_final
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [79]:
#ЯЧЕЙКА 30 — Fine-tuning после MLM
mlm_ner_args = TrainingArguments(
    output_dir="./rubert_after_mlm",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    do_train=True,
    do_eval=True
)

In [80]:
#ЯЧЕЙКА 31 — Trainer после MLM
mlm_ner_trainer = Trainer(
    model=mlm_ner_model,
    args=mlm_ner_args,

    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [81]:
#ЯЧЕЙКА 32 — Обучение после MLM
mlm_ner_trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.109528,0.026520,0.992772,0.994983,0.993876,0.993396
2,0.014687,0.017880,0.995946,0.995474,0.995710,0.995603
3,0.006370,0.020678,0.995058,0.995948,0.995503,0.995520


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2907, training_loss=0.030350202131320572, metrics={'train_runtime': 449.2378, 'train_samples_per_second': 51.728, 'train_steps_per_second': 6.471, 'total_flos': 565674868981068.0, 'train_loss': 0.030350202131320572, 'epoch': 3.0})

In [82]:
#ЯЧЕЙКА 33 — Финальная оценка
mlm_metrics = mlm_ner_trainer.evaluate(
    tokenized_datasets["test"]
)

print(mlm_metrics)

{'eval_loss': 0.020036857575178146, 'eval_precision': 0.9957425876925164, 'eval_recall': 0.994696533261039, 'eval_f1': 0.9952192856053029, 'eval_accuracy': 0.9954686649373939, 'eval_runtime': 9.9619, 'eval_samples_per_second': 259.188, 'eval_steps_per_second': 32.424, 'epoch': 3.0}


In [83]:
#ЯЧЕЙКА 34 — Сравнение всех подходов
results = pd.DataFrame({
    "Baseline": baseline_metrics,
    "Fine-tuned": finetuned_metrics,
    "MLM + Fine-tuned": mlm_metrics
})

results

,Baseline,Fine-tuned,MLM + Fine-tuned
eval_loss,1.967425,0.020568,0.020037
eval_model_preparation_time,0.002600,NaN,NaN
eval_precision,0.231353,0.995508,0.995743
eval_recall,0.180318,0.995205,0.994697
eval_f1,0.202672,0.995357,0.995219
eval_accuracy,0.146188,0.995469,0.995469
eval_runtime,8.899200,9.119900,9.961900
eval_samples_per_second,290.137000,283.117000,259.188000
eval_steps_per_second,36.295000,35.417000,32.424000
epoch,NaN,3.000000,3.000000


In [84]:
#ЯЧЕЙКА 35 — Инференс модели
ner_pipeline = pipeline(
    "token-classification",
    model=mlm_ner_trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

text = """
Президент России Владимир Путин встретился с представителями Газпрома в Москве.
"""

preds = ner_pipeline(text)

preds

[{'entity_group': 'LOC',
  'score': np.float32(0.99972004),
  'word': 'Президент',
  'start': 1,
  'end': 10},
 {'entity_group': 'PER',
  'score': np.float32(0.9989446),
  'word': 'России',
  'start': 11,
  'end': 17},
 {'entity_group': 'ORG',
  'score': np.float32(0.9995646),
  'word': 'Владимир',
  'start': 18,
  'end': 26},
 {'entity_group': 'PER',
  'score': np.float32(0.9995852),
  'word': 'Путин',
  'start': 27,
  'end': 32},
 {'entity_group': 'LOC',
  'score': np.float32(0.99986756),
  'word': 'встретился',
  'start': 33,
  'end': 43},
 {'entity_group': 'LOC',
  'score': np.float32(0.9998963),
  'word': 'с',
  'start': 44,
  'end': 45},
 {'entity_group': 'LOC',
  'score': np.float32(0.99942386),
  'word': 'представителями Газпрома',
  'start': 46,
  'end': 70},
 {'entity_group': 'LOC',
  'score': np.float32(0.9998697),
  'word': 'в',
  'start': 71,
  'end': 72},
 {'entity_group': 'PER',
  'score': np.float32(0.9988304),
  'word': 'Москве',
  'start': 73,
  'end': 79},
 {'entity_

Выводы
Базовая модель без fine-tuning показала низкое качество, так как классификационная голова была случайно инициализирована.
Fine-tuning на factRuEval существенно улучшил качество NER:
выросли precision/recall/F1
модель научилась выделять PER/ORG/LOC сущности.
Дополнительное MLM-дообучение позволило адаптировать encoder к домену корпуса, что дополнительно повысило F1.
Whole Word Masking показал себя лучше обычного token masking, так как для NER важна целостность слов и сущностей.
Возможные улучшения:
больше эпох MLM
larger batch size
CRF layer поверх encoder
использование RuRoBERTa-large
генерация synthetic labels на больших корпусах
ensemble моделей
concept masking для сущностей
Что можно сказать преподавателю

Почему выбран DeepPavlov/rubert-base-cased:

хорошая русскоязычная модель
обучена на русском корпусе
хорошо подходит для token classification
относительно быстро обучается в Colab